# WaveBot v1 — EUR/USD Proof-of-Concept Backtest

**Wave-native multi-timeframe forex trading system**

This notebook runs the complete pipeline:
1. **Clone/Pull** latest code from GitHub
2. **Mount Google Drive** for persistent storage (data, checkpoints, results)
3. **Download OANDA data** (skips if already cached on Drive)
4. **Pre-process wave states** (skips if checkpoint exists)
5. **Run backtest** with spread + slippage simulation
6. **Display results** — equity curve, trade log, Go/No-Go decision
7. **Optimize parameters** (walk-forward grouped search)
8. **Save everything** to Drive for download

> All outputs persist to `/content/drive/MyDrive/wavebot/` — you can close Colab and resume later.

---
## Step 1 — Clone Repo & Pull Latest Code

In [ ]:
import os

REPO_URL = "https://github.com/Unseengap/wavebot-v1.git"
REPO_DIR = "/content/wavebot-v1"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !cd {REPO_DIR} && git pull origin main
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -5
print("\n✅ Code ready.")

---
## Step 2 — Install Dependencies & Mount Google Drive

In [ ]:
!pip install oandapyV20 numpy pandas matplotlib plotly requests pytz pyyaml python-dotenv -q

from google.colab import drive
drive.mount('/content/drive')

# --- Create persistent directory structure on Drive ---
DRIVE_BASE = "/content/drive/MyDrive/wavebot"
DRIVE_DATA = f"{DRIVE_BASE}/data"
DRIVE_CHECKPOINTS = f"{DRIVE_BASE}/checkpoints"
DRIVE_RESULTS = f"{DRIVE_BASE}/results"
DRIVE_MODELS = f"{DRIVE_BASE}/models"

for d in [DRIVE_DATA, DRIVE_CHECKPOINTS, DRIVE_RESULTS, DRIVE_MODELS]:
    os.makedirs(d, exist_ok=True)

print(f"Drive base:        {DRIVE_BASE}")
print(f"Data dir:          {DRIVE_DATA}")
print(f"Checkpoints dir:   {DRIVE_CHECKPOINTS}")
print(f"Results dir:       {DRIVE_RESULTS}")
print(f"Models dir:        {DRIVE_MODELS}")
print("\n✅ Drive mounted & directories ready.")

---
## Step 3 — OANDA Credentials

Enter your OANDA Practice API token and account ID below.  
These are **not** saved to Drive or committed to git.

In [ ]:
# Option A: Manual entry (uncomment and fill in)
# OANDA_TOKEN = "your_64_character_token_here"
# OANDA_ACCOUNT = "your_account_id_here"

# Option B: Colab secrets (recommended — go to key icon in left sidebar)
try:
    from google.colab import userdata
    OANDA_TOKEN = userdata.get('OANDA_TOKEN')
    OANDA_ACCOUNT = userdata.get('OANDA_ACCOUNT')
    print("✅ Loaded credentials from Colab Secrets.")
except:
    # Fallback — set manually above
    if 'OANDA_TOKEN' not in dir() or 'your_' in OANDA_TOKEN:
        print("⚠️  Set OANDA_TOKEN and OANDA_ACCOUNT above, or add them to Colab Secrets.")
        print("   (Left sidebar → Key icon → Add OANDA_TOKEN and OANDA_ACCOUNT)")
    else:
        print("✅ Using manually entered credentials.")

INSTRUMENT = "EUR_USD"
TIMEFRAMES = ["M5", "M15", "H1", "H4", "D"]
DATA_START = "2020-01-01T00:00:00Z"
DATA_END = "2024-12-31T00:00:00Z"

print(f"\nInstrument: {INSTRUMENT}")
print(f"Timeframes: {TIMEFRAMES}")
print(f"Period:     {DATA_START[:10]} to {DATA_END[:10]}")

---
## Step 4 — Download OANDA Data (Skips if Cached)

Data is saved as Parquet files on Drive.  
If the file already exists for a timeframe, it's loaded from cache — no re-download.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import pandas as pd
import numpy as np
from src.data.oanda_client import OandaClient

client = OandaClient(OANDA_TOKEN, OANDA_ACCOUNT, environment="practice")

candle_data = {}  # tf -> DataFrame

for tf in TIMEFRAMES:
    cache_file = f"{DRIVE_DATA}/{INSTRUMENT}_{tf}.parquet"

    if os.path.exists(cache_file):
        df = pd.read_parquet(cache_file)
        print(f"  {tf}: Loaded from cache — {len(df):,} candles  ({cache_file.split('/')[-1]})")
    else:
        print(f"  {tf}: Downloading from OANDA...")
        df = client.collect_range(INSTRUMENT, tf, start=DATA_START, end=DATA_END)
        df.to_parquet(cache_file, index=False)
        print(f"  {tf}: Downloaded {len(df):,} candles → saved to Drive")

    candle_data[tf] = df

print(f"\n✅ All data ready:")
for tf, df in candle_data.items():
    print(f"   {tf:>4}: {len(df):>8,} candles  |  {str(df['time'].iloc[0])[:10]} → {str(df['time'].iloc[-1])[:10]}")

---
## Step 5 — Run Backtest (Default Parameters)

First run uses the **balanced** parameter set:  
- `min_confluence_score = 0.65` (aggressive — ensures 2-10 trades/day)  
- `min_frames_aligned = 2`  
- `min_rr = 2.0`, `tp_percentile = p75`  
- Full spread + slippage simulation  
- `$10,000` starting balance, 1% risk per trade

Results checkpoint to Drive — if you rerun, it loads from cache.

In [ ]:
import json
import pickle
from src.backtest.engine import BacktestEngine
from src.backtest.metrics import calculate_metrics, print_metrics

# --- Configuration ---
config = {
    "instrument": INSTRUMENT,
    "initial_balance": 10000.0,
    "risk_fraction": 0.01,
    "start": DATA_START[:10],
    "end": DATA_END[:10],
    # Confluence (aggressive to hit 2-10 trades/day)
    "min_confluence_score": 0.65,
    "min_frames_aligned": 2,
    # Risk
    "min_rr_ratio": 2.0,
    "tp_percentile": "p75",
    "sl_buffer_pips": 2.0,
    "max_entry_maturity": 0.75,
    # Limits
    "max_daily_drawdown": 0.02,
    "max_total_drawdown": 0.08,
    "max_open_trades": 3,
}

# --- Check for cached results ---
checkpoint_file = f"{DRIVE_CHECKPOINTS}/backtest_default_{INSTRUMENT}.pkl"

if os.path.exists(checkpoint_file):
    print("Loading cached backtest results from Drive...")
    with open(checkpoint_file, 'rb') as f:
        cached = pickle.load(f)
    trades = cached['trades']
    metrics = cached['metrics']
    print(f"✅ Loaded {len(trades)} trades from checkpoint.")
else:
    print("No checkpoint found — running fresh backtest...")
    print("="*60)

    engine = BacktestEngine(config)
    trades = engine.run(candle_data)

    # Calculate trading days
    m5_df = candle_data["M5"]
    trading_days = (pd.Timestamp(m5_df['time'].iloc[-1]) -
                    pd.Timestamp(m5_df['time'].iloc[0])).days
    trading_days = max(1, trading_days)

    metrics = calculate_metrics(
        trades, config['initial_balance'],
        INSTRUMENT, config['start'], config['end'],
        trading_days
    )

    # Save checkpoint to Drive
    with open(checkpoint_file, 'wb') as f:
        pickle.dump({'trades': trades, 'metrics': metrics, 'config': config}, f)
    print(f"\n💾 Checkpoint saved to Drive.")

print()
print_metrics(metrics)

---
## Step 6 — Equity Curve & Trade Analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle(f'WaveBot Backtest — {INSTRUMENT}  |  ${config["initial_balance"]:,.0f} Start', fontsize=14, fontweight='bold')

# --- 1. Equity Curve ---
ax = axes[0, 0]
equity = metrics.equity_curve
ax.plot(equity, color='#2962FF', linewidth=1.2)
ax.axhline(y=config['initial_balance'], color='gray', linestyle='--', alpha=0.5)
ax.fill_between(range(len(equity)), config['initial_balance'], equity,
                where=[e > config['initial_balance'] for e in equity],
                alpha=0.15, color='green')
ax.fill_between(range(len(equity)), config['initial_balance'], equity,
                where=[e < config['initial_balance'] for e in equity],
                alpha=0.15, color='red')
ax.set_title('Equity Curve')
ax.set_ylabel('Balance ($)')
ax.grid(True, alpha=0.3)

# --- 2. Drawdown ---
ax = axes[0, 1]
eq_arr = np.array(equity)
peak = np.maximum.accumulate(eq_arr)
dd_pct = (peak - eq_arr) / peak * 100
ax.fill_between(range(len(dd_pct)), 0, dd_pct, color='red', alpha=0.4)
ax.set_title('Drawdown (%)')
ax.set_ylabel('Drawdown %')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

# --- 3. P&L per Trade ---
ax = axes[1, 0]
pnls = [t['pnl_pips'] for t in trades]
colors = ['green' if p > 0 else 'red' for p in pnls]
ax.bar(range(len(pnls)), pnls, color=colors, width=1.0, alpha=0.7)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('P&L per Trade (pips)')
ax.set_ylabel('Pips')
ax.grid(True, alpha=0.3)

# --- 4. Cumulative Pips ---
ax = axes[1, 1]
cum_pips = np.cumsum(pnls)
ax.plot(cum_pips, color='#2962FF', linewidth=1.2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Cumulative Pips')
ax.set_ylabel('Pips')
ax.grid(True, alpha=0.3)

# --- 5. Win/Loss Distribution ---
ax = axes[2, 0]
ax.hist(pnls, bins=50, color='#2962FF', alpha=0.7, edgecolor='white')
ax.axvline(x=0, color='red', linewidth=1)
ax.axvline(x=np.mean(pnls), color='green', linewidth=1.5, linestyle='--',
           label=f'Mean: {np.mean(pnls):.1f} pips')
ax.set_title('P&L Distribution')
ax.set_xlabel('Pips')
ax.legend()
ax.grid(True, alpha=0.3)

# --- 6. Trades by Session ---
ax = axes[2, 1]
sessions = {}
for t in trades:
    s = t.get('session', 'Unknown')
    sessions.setdefault(s, {'wins': 0, 'losses': 0})
    if t['pnl_pips'] > 0:
        sessions[s]['wins'] += 1
    else:
        sessions[s]['losses'] += 1

if sessions:
    labels = list(sessions.keys())
    wins = [sessions[s]['wins'] for s in labels]
    losses = [sessions[s]['losses'] for s in labels]
    x = range(len(labels))
    ax.bar(x, wins, color='green', alpha=0.7, label='Wins')
    ax.bar(x, losses, bottom=wins, color='red', alpha=0.7, label='Losses')
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_title('Trades by Session')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DRIVE_RESULTS}/backtest_charts_{INSTRUMENT}.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💾 Charts saved to Drive.")

---
## Step 7 — Trade Log (Sample)

In [ ]:
# Show first 20 and last 20 trades
trade_df = pd.DataFrame(trades)
display_cols = ['id', 'direction', 'entry_time', 'exit_time', 'entry_price',
                'exit_price', 'stop_loss', 'take_profit', 'pnl_pips',
                'pnl_dollars', 'exit_reason', 'confluence_score', 'session', 'signal_frame']
available_cols = [c for c in display_cols if c in trade_df.columns]

print(f"Total trades: {len(trade_df)}")
print(f"\n--- First 20 trades ---")
display(trade_df[available_cols].head(20))

print(f"\n--- Last 20 trades ---")
display(trade_df[available_cols].tail(20))

# Save full trade log
trade_df.to_csv(f"{DRIVE_RESULTS}/trades_{INSTRUMENT}.csv", index=False)
print(f"\n💾 Full trade log saved: trades_{INSTRUMENT}.csv")

---
## Step 8 — Go / No-Go Decision

In [ ]:
print("=" * 65)
print("  GO / NO-GO DECISION MATRIX")
print("=" * 65)

checks = {
    "Win Rate ≥ 50%":      (metrics.win_rate >= 0.50,  f"{metrics.win_rate:.1%}"),
    "Profit Factor ≥ 1.3": (metrics.profit_factor >= 1.3, f"{metrics.profit_factor:.2f}"),
    "Max Drawdown < 15%":  (metrics.max_drawdown_pct < 15, f"{metrics.max_drawdown_pct:.2f}%"),
    "Sharpe Ratio ≥ 1.0":  (metrics.sharpe_ratio >= 1.0, f"{metrics.sharpe_ratio:.2f}"),
    "Trades/Day ≥ 2":      (metrics.trades_per_day >= 2.0, f"{metrics.trades_per_day:.1f}"),
}

all_pass = True
for name, (passed, value) in checks.items():
    icon = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"  {icon} {name:<28} {value:>10}   [{status}]")
    if not passed:
        all_pass = False

print("=" * 65)
if all_pass:
    print("  ✅ DECISION: GO — All checks pass. Proceed to paper trading.")
else:
    failing = sum(1 for _, (p, _) in checks.items() if not p)
    if failing <= 2:
        print("  ⚠️  DECISION: CAUTION — Some checks failing. Review before proceeding.")
    else:
        print("  ❌ DECISION: NO-GO — Model needs work. Do not paper trade yet.")
print("=" * 65)

---
## Step 9 — Parameter Optimization (Walk-Forward Grouped Search)

Searches **108 parameter combinations** (3 wave × 3 confluence × 3 risk × 4 maturity).  
Checkpoints after each combination — safe to interrupt and resume.

In [ ]:
from src.backtest.optimizer import (
    build_param_grid, run_single_backtest, score_metrics, expand_params
)

# --- Check for optimization checkpoint ---
opt_checkpoint = f"{DRIVE_CHECKPOINTS}/optimization_progress_{INSTRUMENT}.pkl"
opt_results = []
start_idx = 0

if os.path.exists(opt_checkpoint):
    with open(opt_checkpoint, 'rb') as f:
        opt_state = pickle.load(f)
    opt_results = opt_state.get('results', [])
    start_idx = len(opt_results)
    print(f"Resuming optimization from combination {start_idx}/108...")
else:
    print("Starting fresh optimization...")

grid = build_param_grid()
total = len(grid)
print(f"Total combinations: {total}")
print(f"Already completed:  {start_idx}")
print(f"Remaining:          {total - start_idx}")
print("=" * 60)

base_config = {
    "instrument": INSTRUMENT,
    "initial_balance": 10000.0,
    "risk_fraction": 0.01,
    "start": DATA_START[:10],
    "end": DATA_END[:10],
    "max_daily_drawdown": 0.02,
    "max_total_drawdown": 0.08,
    "max_open_trades": 3,
}

CHECKPOINT_EVERY = 5  # Save to Drive every N combinations

for idx in range(start_idx, total):
    params = grid[idx]
    label = f"{params['wave_sensitivity']}/{params['confluence_strictness']}/{params['risk_profile']}/mat{params['max_entry_maturity']}"

    try:
        m, t = run_single_backtest(candle_data, base_config, params)
        s = score_metrics(m)

        result = {
            'idx': idx,
            'params': params,
            'score': s,
            'trades': m.total_trades,
            'win_rate': m.win_rate,
            'pf': m.profit_factor,
            'sharpe': m.sharpe_ratio,
            'max_dd': m.max_drawdown_pct,
            'pips': m.total_pips,
            'trades_per_day': m.trades_per_day,
            'final_balance': m.final_balance,
        }
        opt_results.append(result)

        print(f"  [{idx+1:>3}/{total}] {label:<50} | "
              f"Score: {s:>6.3f} | WR: {m.win_rate:.0%} | PF: {m.profit_factor:.2f} | "
              f"Sharpe: {m.sharpe_ratio:.2f} | Trades: {m.total_trades:>5} | "
              f"${m.final_balance:>10,.2f}")

    except Exception as e:
        print(f"  [{idx+1:>3}/{total}] {label:<50} | ERROR: {str(e)[:60]}")
        opt_results.append({'idx': idx, 'params': params, 'score': -999, 'error': str(e)})

    # Checkpoint to Drive
    if (idx + 1) % CHECKPOINT_EVERY == 0 or idx == total - 1:
        with open(opt_checkpoint, 'wb') as f:
            pickle.dump({'results': opt_results}, f)

print("\n✅ Optimization complete. Saved to Drive.")

---
## Step 10 — Optimization Results & Best Parameters

In [ ]:
# Rank results
valid_results = [r for r in opt_results if r.get('score', -999) > -999]
ranked = sorted(valid_results, key=lambda x: x['score'], reverse=True)

print("=" * 90)
print("  TOP 10 PARAMETER SETS")
print("=" * 90)
print(f"  {'Rank':<5} {'Wave':<8} {'Confluence':<14} {'Risk':<10} {'Mat':<5} | "
      f"{'Score':<7} {'WR':<6} {'PF':<6} {'Sharpe':<7} {'Trades':<7} {'T/Day':<6} {'Final$':<12}")
print("-" * 90)

for i, r in enumerate(ranked[:10]):
    p = r['params']
    print(f"  {i+1:<5} {p['wave_sensitivity']:<8} {p['confluence_strictness']:<14} "
          f"{p['risk_profile']:<10} {p['max_entry_maturity']:<5.2f} | "
          f"{r['score']:<7.3f} {r['win_rate']:<6.0%} {r['pf']:<6.2f} "
          f"{r['sharpe']:<7.2f} {r['trades']:<7} {r['trades_per_day']:<6.1f} "
          f"${r['final_balance']:<11,.2f}")

# Save best params as the "model"
if ranked:
    best = ranked[0]
    print("\n" + "=" * 90)
    print(f"  BEST: {best['params']}")
    print(f"  Score: {best['score']:.3f} | WR: {best['win_rate']:.1%} | "
          f"PF: {best['pf']:.2f} | Sharpe: {best['sharpe']:.2f} | "
          f"Trades/Day: {best['trades_per_day']:.1f}")
    print("=" * 90)

    # Save best model to Drive
    model_data = {
        'best_params': best['params'],
        'expanded_params': expand_params(
            best['params']['wave_sensitivity'],
            best['params']['confluence_strictness'],
            best['params']['risk_profile'],
            best['params']['max_entry_maturity'],
        ),
        'metrics': {
            'score': best['score'],
            'win_rate': best['win_rate'],
            'profit_factor': best['pf'],
            'sharpe': best['sharpe'],
            'max_drawdown': best['max_dd'],
            'total_trades': best['trades'],
            'trades_per_day': best['trades_per_day'],
            'final_balance': best['final_balance'],
        },
        'instrument': INSTRUMENT,
        'data_period': f"{DATA_START[:10]} to {DATA_END[:10]}",
        'all_results': ranked,
    }

    with open(f"{DRIVE_MODELS}/best_model_{INSTRUMENT}.pkl", 'wb') as f:
        pickle.dump(model_data, f)

    with open(f"{DRIVE_MODELS}/best_model_{INSTRUMENT}.json", 'w') as f:
        json.dump({
            'best_params': best['params'],
            'expanded_params': model_data['expanded_params'],
            'metrics': model_data['metrics'],
            'instrument': INSTRUMENT,
            'data_period': model_data['data_period'],
        }, f, indent=2)

    print(f"\n💾 Best model saved to Drive:")
    print(f"   {DRIVE_MODELS}/best_model_{INSTRUMENT}.pkl")
    print(f"   {DRIVE_MODELS}/best_model_{INSTRUMENT}.json")

---
## Step 11 — Rerun Backtest with Best Parameters

Uses the winning parameter set from optimization to produce final proof results.

In [ ]:
# Load best model if exists
best_model_file = f"{DRIVE_MODELS}/best_model_{INSTRUMENT}.pkl"

if os.path.exists(best_model_file):
    with open(best_model_file, 'rb') as f:
        best_model = pickle.load(f)

    best_params = best_model['best_params']
    print(f"Loaded best model: {best_params}")
    print(f"Running final backtest with optimized parameters...\n")

    final_metrics, final_trades = run_single_backtest(
        candle_data, base_config, best_params
    )

    print_metrics(final_metrics)

    # Save final results
    final_output = {
        'metrics': final_metrics,
        'trades': final_trades,
        'params': best_params,
        'config': base_config,
    }
    with open(f"{DRIVE_RESULTS}/final_backtest_{INSTRUMENT}.pkl", 'wb') as f:
        pickle.dump(final_output, f)

    final_trade_df = pd.DataFrame(final_trades)
    final_trade_df.to_csv(f"{DRIVE_RESULTS}/final_trades_{INSTRUMENT}.csv", index=False)

    print(f"\n💾 Final results saved to Drive.")
else:
    print("No optimized model found. Run Step 9 first.")

---
## Step 12 — Final Equity Curve (Optimized)

In [ ]:
if 'final_metrics' in dir() and final_metrics.equity_curve:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle(f'WaveBot OPTIMIZED — {INSTRUMENT}  |  '
                 f'${config["initial_balance"]:,.0f} → ${final_metrics.final_balance:,.2f}  '
                 f'({final_metrics.total_return_pct:+.1f}%)',
                 fontsize=13, fontweight='bold')

    # Equity
    eq = final_metrics.equity_curve
    ax1.plot(eq, color='#2962FF', linewidth=1.2)
    ax1.axhline(y=config['initial_balance'], color='gray', linestyle='--', alpha=0.4)
    ax1.fill_between(range(len(eq)), config['initial_balance'], eq,
                     where=[e > config['initial_balance'] for e in eq],
                     alpha=0.12, color='green')
    ax1.fill_between(range(len(eq)), config['initial_balance'], eq,
                     where=[e < config['initial_balance'] for e in eq],
                     alpha=0.12, color='red')
    ax1.set_ylabel('Balance ($)')
    ax1.grid(True, alpha=0.3)

    # Drawdown
    eq_arr = np.array(eq)
    peak = np.maximum.accumulate(eq_arr)
    dd = (peak - eq_arr) / peak * 100
    ax2.fill_between(range(len(dd)), 0, dd, color='red', alpha=0.4)
    ax2.set_ylabel('Drawdown %')
    ax2.set_xlabel('Trade #')
    ax2.invert_yaxis()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{DRIVE_RESULTS}/final_equity_{INSTRUMENT}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Final chart saved to Drive.")
else:
    print("No final metrics available. Run Step 11 first.")

---
## Step 13 — Summary & Drive Contents

Everything is persisted on your Google Drive at `/MyDrive/wavebot/`.  
You can download locally from there.

In [ ]:
print("=" * 65)
print("  FILES SAVED TO GOOGLE DRIVE")
print("=" * 65)

for root, dirs, files in os.walk(DRIVE_BASE):
    level = root.replace(DRIVE_BASE, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    print(f"{indent}📁 {folder}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        if size_mb > 1:
            print(f"{sub_indent}📄 {f}  ({size_mb:.1f} MB)")
        else:
            size_kb = os.path.getsize(fpath) / 1024
            print(f"{sub_indent}📄 {f}  ({size_kb:.0f} KB)")

print("\n" + "=" * 65)
print(f"  Download from: Google Drive → My Drive → wavebot/")
print("=" * 65)
print("\n🏁 All done. To resume later, just rerun this notebook —")
print("   cached data and checkpoints will be loaded automatically.")